In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FE_rain") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
df = spark.read.csv(
    "hdfs://localhost:9000/DACK/weather_clean.csv",
    header=True,
    inferSchema=True,
    nullValue="NA"
)

In [4]:
df.printSchema()


root
 |-- Date: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- MinTemp: double (nullable = true)
 |-- MaxTemp: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: double (nullable = true)
 |-- WindGustDir: string (nullable = true)
 |-- WindGustSpeed: integer (nullable = true)
 |-- WindDir9am: string (nullable = true)
 |-- WindDir3pm: string (nullable = true)
 |-- WindSpeed9am: integer (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = true)
 |-- Pressure3pm: double (nullable = true)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = true)
 |-- Temp3pm: double (nullable = true)
 |-- RainToday: string (nullable = true)
 |-- RainTomorrow: string (nullable = true)
 |-- label: double (nullable = true)

In [5]:
from pyspark.sql.functions import *

df = df.withColumn(
    "TempRange",
    col("MaxTemp") - col("MinTemp")
)

In [6]:
df = df.withColumn(
    "HumidityDiff",
    col("Humidity9am") - col("Humidity3pm")
)

In [7]:
df = df.withColumn(
    "PressureDiff",
    col("Pressure9am") - col("Pressure3pm")
)

In [8]:
df = df.withColumn(
    "WindSpeedDiff",
    col("WindSpeed3pm") - col("WindSpeed9am")
)

In [9]:
feature_cols = [

    # Temperature
    "MinTemp",
    "MaxTemp",
    "Temp9am",
    "Temp3pm",
    "TempRange",

    # Rain
    "Rainfall",

    # Humidity
    "Humidity9am",
    "Humidity3pm",
    "HumidityDiff",

    # Pressure
    "Pressure9am",
    "Pressure3pm",
    "PressureDiff",

    # Wind
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "WindSpeedDiff",

    # Time
    "Month",

    # Encoded categorical
    "Location_index",
    "WindGustDir_index",
    "WindDir9am_index",
    "WindDir3pm_index",
    "RainToday_index"
]

In [10]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = assembler.transform(df)

In [11]:
df.select(
    "features",
    "label"
).show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                 |label|
+-----------------------------------------------------------------------------------------------------------------------------------------+-----+
|[13.4,22.9,16.9,21.8,9.499999999999998,0.6,71.0,22.0,49.0,1007.7,1007.1,0.6000000000000227,44.0,20.0,24.0,4.0,12.0,9.0,1.0,7.0,7.0,0.0]  |0.0  |
|[7.4,25.1,17.2,24.3,17.700000000000003,0.0,44.0,25.0,19.0,1010.6,1007.8,2.800000000000068,44.0,4.0,22.0,18.0,12.0,9.0,10.0,10.0,3.0,0.0] |0.0  |
|[12.9,25.7,21.0,23.2,12.799999999999999,0.0,38.0,30.0,8.0,1007.6,1008.7,-1.1000000000000227,46.0,19.0,26.0,7.0,12.0,9.0,7.0,7.0,3.0,0.0] |0.0  |
|[9.2,28.0,18.1,26.5,18.8,0.0,45.0,16.0,29.0,1017.6,1012.8,4.800000000000068,24.0,11.0,9.0,-2.0,12.0,9.0,14.0,2.0,10.0,0.0] 

In [12]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(df)

df = scaler_model.transform(df)

In [13]:
final_df = df.select(
    "scaled_features",
    "label"
)

final_df.show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|scaled_features                                                                                                                                                                                                                                                                                                                                                                                             |label|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
df.show(5)

+---------+--------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-----+--------------+-----------------+----------------+----------------+---------------+-------------------+-----+----+------------------+------------+-------------------+-------------+--------------------+--------------------+
|     Date|Location|MinTemp|MaxTemp|Rainfall|Evaporation|Sunshine|WindGustDir|WindGustSpeed|WindDir9am|WindDir3pm|WindSpeed9am|WindSpeed3pm|Humidity9am|Humidity3pm|Pressure9am|Pressure3pm|Cloud9am|Cloud3pm|Temp9am|Temp3pm|RainToday|RainTomorrow|label|Location_index|WindGustDir_index|WindDir9am_index|WindDir3pm_index|RainToday_index|         DateParsed|Month|Year|         TempRange|HumidityDiff|       PressureDiff|WindSpeedDiff|            features|     scaled_features|
+---------+--------+-------+-------+--------+-------

In [17]:
final_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet("weather_ml_rain")